# 🚀 UnifiedDiffusionPlanner Training

Production notebook for training the diffusion-based robot manipulation policy.

**Runtime**: Select GPU (T4/A100/L4)

## 1️⃣ GPU Rendering Setup (Run First!)

In [ ]:
# CELL 1: EGL Setup - MUST RUN BEFORE ANY OTHER IMPORTS
import os
import subprocess
import logging
import sys

log = logging.getLogger("GPU_Setup")
log.setLevel(logging.INFO)
if not log.hasHandlers():
    log.addHandler(logging.StreamHandler(sys.stdout))

IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    log.info("🚀 Colab environment detected. Initializing High-Performance EGL Setup...")

    # Set Env Var (Essential for MuJoCo to pick up EGL)
    os.environ['MUJOCO_GL'] = 'egl'

    # Fast Library Search using ldconfig
    log.info("🔍 Locating NVIDIA EGL drivers via ldconfig cache...")
    try:
        result = subprocess.run(["ldconfig", "-p"], capture_output=True, text=True, check=True)
        nvidia_lib_line = [line for line in result.stdout.split('\n') if "libEGL_nvidia.so.0" in line]

        if nvidia_lib_line:
            nvidia_lib_path = nvidia_lib_line[0].split("=>")[1].strip()
            log.info(f"✅ Found driver (Cache Hit): {nvidia_lib_path}")

            # Write EGL Configuration
            icd_path = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
            icd_content = f'{{\n    "file_format_version" : "1.0.0",\n    "ICD": {{\n        "library_path": "{nvidia_lib_path}"\n    }}\n}}'

            os.makedirs(os.path.dirname(icd_path), exist_ok=True)
            with open(icd_path, 'w') as f:
                f.write(icd_content)
            log.info("✅ NVIDIA EGL Vendor Configured.")
        else:
            log.warning("⚠️ Cache Miss. Falling back to disk scan...")
            lib_path_process = subprocess.run("find /usr/ -name libEGL_nvidia.so.0 2>/dev/null", shell=True, capture_output=True, text=True)
            nvidia_lib_path = lib_path_process.stdout.strip().split('\n')[0]
            if nvidia_lib_path:
                icd_path = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
                icd_content = f'{{\n    "file_format_version" : "1.0.0",\n    "ICD": {{\n        "library_path": "{nvidia_lib_path}"\n    }}\n}}'
                os.makedirs(os.path.dirname(icd_path), exist_ok=True)
                with open(icd_path, 'w') as f:
                    f.write(icd_content)
                log.info(f"✅ Found and configured: {nvidia_lib_path}")

    except Exception as e:
        log.error(f"❌ GPU Setup Failed: {e}")
        log.warning("   Falling back to CPU Rendering.")
        if 'MUJOCO_GL' in os.environ:
            del os.environ['MUJOCO_GL']

    # Install Dependencies
    log.info("📦 Verifying GL dependencies...")
    try:
        subprocess.run(['apt-get', 'install', '-y', '-qq', 'libegl1', 'patchelf'], check=True, capture_output=True)
    except:
        pass

else:
    log.info("ℹ️ Local environment detected. Skipping EGL setup.")

# Import Verification
try:
    import mujoco
    if IN_COLAB and os.environ.get('MUJOCO_GL') == 'egl':
        model = mujoco.MjModel.from_xml_string('<mujoco/>')
        data = mujoco.MjData(model)
        renderer = mujoco.Renderer(model, height=64, width=64)
        renderer.update_scene(data)
        log.info("✅ EGL Context Created Successfully. Hardware Rendering Active.")
except Exception as e:
    log.warning(f"⚠️ MuJoCo render test: {e}")

## 2️⃣ Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create output directories
!mkdir -p /content/drive/MyDrive/pda/models/checkpoints
!mkdir -p /content/drive/MyDrive/pda/logs
print("✅ Drive mounted")

## 3️⃣ Clone Repository

In [ ]:
!rm -rf /content/dgpo
!git clone -b aswasp https://github.com/Luke23-45/dgpo.git /content/dgpo
%cd /content/dgpo

## 4️⃣ Install Dependencies

In [ ]:
!pip install -q \
    "mujoco>=3.0.0" \
    gymnasium numpy scipy pyyaml tqdm \
    ikpy lmdb hydra-core omegaconf \
    diffusers transformers pytorch_lightning \
    tensorboard wandb huggingface_hub lpips

print("✅ Dependencies installed")

## 5️⃣ Download Training Data from HuggingFace

In [ ]:
# === CONFIGURE YOUR HUGGINGFACE DATASET ===
HF_DATASET_REPO = "Luke23-45/panda-pick-place"  # <-- UPDATE THIS
DATA_DIR = "/content/data"

from huggingface_hub import snapshot_download
import os

print(f"📦 Downloading dataset from: {HF_DATASET_REPO}")
os.makedirs(DATA_DIR, exist_ok=True)

try:
    snapshot_download(
        repo_id=HF_DATASET_REPO,
        repo_type="dataset",
        local_dir=DATA_DIR,
        local_dir_use_symlinks=False
    )
    print("✅ Dataset downloaded!")
except Exception as e:
    print(f"⚠️ Download failed: {e}")
    print("   Run: huggingface-cli login")

# Show files
!ls -la {DATA_DIR}

In [ ]:
# Verify LMDB files
import os

for root, dirs, files in os.walk(DATA_DIR):
    for f in files:
        if f.endswith('.lmdb'):
            path = os.path.join(root, f)
            size_gb = os.path.getsize(path) / (1024**3)
            print(f"📁 {path}: {size_gb:.2f} GB")

## 6️⃣ Configure Training Paths

**Update `TRAIN_PATH` below to match your downloaded data!**

In [ ]:
# === UPDATE THESE PATHS ===
TRAIN_PATH = "/content/data/training_set.lmdb"  # <-- UPDATE THIS
VAL_PATH = None  # Optional: "/content/data/val_set.lmdb"
PRETRAINED_CKPT = "/content/drive/MyDrive/pda/models/backups/unified_planner_initialized.ckpt"  # Optional

# Verify paths exist
import os
if os.path.exists(TRAIN_PATH):
    print(f"✅ Train path valid: {TRAIN_PATH}")
else:
    print(f"❌ Train path NOT FOUND: {TRAIN_PATH}")
    print("   Update TRAIN_PATH above!")

## 7️⃣ Start Training

In [ ]:
# Build training command
cmd = f"""python train/train_unified_planner.py \
    --config-name train_unified_planner_config \
    dataset.train_path={TRAIN_PATH} \
    logging.output_dir=/content/drive/MyDrive/pda/models"""

if VAL_PATH:
    cmd += f" dataset.val_path={VAL_PATH}"

if PRETRAINED_CKPT and os.path.exists(PRETRAINED_CKPT):
    cmd += f" training.pretrained_checkpoint={PRETRAINED_CKPT}"

print(f"🚀 Running:\n{cmd}\n")
!{cmd}

## 📊 Monitor Training (Optional)

In [ ]:
# TensorBoard
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/pda/logs

In [ ]:
# Sync WandB (after training)
!wandb sync /content/drive/MyDrive/pda/logs/wandb/offline-run-*

## 📦 Checkpoints

In [ ]:
!ls -la /content/drive/MyDrive/pda/models/checkpoints/

---

## 🔧 Alternative: Generate Data Locally

If you don't have HuggingFace data, generate it:

In [ ]:
# Generate training data (only if needed)
# !python scripts/bootstrap_training_data.py --episodes 400 --workers 2 --output_dir /content/data/